# Full Demo Integration Test

Run the production state machine without parameter editing controls. Edit JSON on disk, reload config, then test one step or the complete mission.

In [ ]:
from __future__ import print_function

import contextlib
import io
import os
import sys
import time
import traceback

def find_project_root():
    current = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isfile(os.path.join(current, 'config.json')):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError('config.json not found')

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import ipywidgets as widgets
from IPython.display import Javascript, display

from demo_core import DemoDiagnostics, DemoStateMachine, load_config
from demo_core.logging_utils import default_log_path

disk_config = load_config()
runtime = None
output = widgets.Output(layout={'border': '1px solid #ccc', 'height': '560px', 'overflow_y': 'auto'})
reload_models = widgets.Checkbox(value=False, description='reload_models')
compact_log = widgets.Checkbox(value=False, description='compact_log')
camera_real = widgets.Checkbox(value=not disk_config.get('runtime.dry_run.camera', True), description='camera_real')
base_real = widgets.Checkbox(value=not disk_config.get('runtime.dry_run.base', True), description='base_real')
arm_real = widgets.Checkbox(value=not disk_config.get('runtime.dry_run.arm', True), description='arm_real')

class Tee(object):
    def __init__(self, *streams): self.streams = streams
    def write(self, data):
        for stream in self.streams: stream.write(data); stream.flush()
    def flush(self):
        for stream in self.streams: stream.flush()

def scroll_log():
    display(Javascript("setTimeout(function(){var x=document.querySelectorAll('.widget-output');for(var i=0;i<x.length;i++){x[i].scrollTop=x[i].scrollHeight;}},80);"))

def important(line):
    tokens = ('[fsm]', '[approach]', '[mission]', '[base]', '[arm]', '[can]', '[tag]', '[depth]', '[error]', 'Traceback', 'FAILED', 'DONE')
    return any(token in line for token in tokens)

def logged(fn):
    def wrapped(_=None):
        with output:
            path = default_log_path('integration_{}'.format(fn.__name__))
            with open(path, 'a') as log_file:
                tee = Tee(sys.stdout, log_file)
                with contextlib.redirect_stdout(tee), contextlib.redirect_stderr(tee):
                    start = time.time(); status = 'SUCCESS'
                    try:
                        print('\n>>> {} log={}'.format(fn.__name__, path))
                        if compact_log.value:
                            capture = io.StringIO()
                            with contextlib.redirect_stdout(capture), contextlib.redirect_stderr(capture): result = fn()
                            for line in capture.getvalue().splitlines():
                                if important(line): print(line)
                        else:
                            result = fn()
                        print('[result] {}'.format(result))
                    except Exception as exc:
                        status = 'FAILED'; print('[error] {}'.format(exc)); traceback.print_exc()
                    print('[integration] {} elapsed={:.1f}s'.format(status, time.time() - start))
            scroll_log()
    return wrapped

def overrides():
    return {'runtime': {'dry_run': {'camera': not camera_real.value, 'base': not base_real.value, 'arm': not arm_real.value}}}

def reload_config():
    global disk_config, runtime
    if runtime is not None:
        runtime.release_camera()
    disk_config = load_config(overrides=overrides())
    runtime = DemoStateMachine(disk_config)
    return {'config': disk_config.parameters_path, 'dry_run': disk_config.get('runtime.dry_run')}

def current():
    global runtime
    if runtime is None:
        reload_config()
    return runtime

def show_state():
    rt = current()
    return {'state': rt.state.value, 'context': rt.context.snapshot()}

def start_camera():
    current().start_camera()
    return show_state()

def load_models():
    current().load_detectors(bool(reload_models.value))
    return show_state()

def preflight():
    diag = DemoDiagnostics(current().config, services=current().services)
    return diag.preflight(bool(reload_models.value))

def step_once():
    outcome = current().step_once()
    return {'state': current().state.value, 'event': outcome.event.value if outcome.event else None, 'reason': outcome.reason}

def run_full():
    return current().run()

def pause(): current().request_pause(); return show_state()
def resume(): current().resume(); return show_state()
def stop_all(): current().request_stop(); current().stop_all(); return show_state()
def release_camera(): current().release_camera(); return show_state()
def reset_runtime():
    global runtime
    if runtime is not None: runtime.stop_all()
    runtime = None
    return True
def clear_log(): output.clear_output(); return True

items = [('Reload Config', reload_config), ('Show State', show_state), ('Start Camera', start_camera), ('Load Models', load_models), ('Preflight', preflight), ('Step Once', step_once), ('Run Full Demo', run_full), ('Pause', pause), ('Resume', resume), ('STOP ALL', stop_all), ('Release Camera', release_camera), ('Reset Runtime', reset_runtime), ('Clear Log', clear_log)]
buttons = []
for label, fn in items:
    button = widgets.Button(description=label, layout=widgets.Layout(width='150px'))
    if 'STOP' in label: button.button_style = 'danger'
    if label == 'Run Full Demo': button.button_style = 'success'
    button.on_click(logged(fn)); buttons.append(button)
display(widgets.VBox([widgets.HBox([reload_models, compact_log, camera_real, base_real, arm_real]), widgets.HBox(buttons[0:4]), widgets.HBox(buttons[4:7]), widgets.HBox(buttons[7:10]), widgets.HBox(buttons[10:13]), output]))
print('[integration] ready; edit JSON, then Reload Config')
